# PortWatch AI — ML Setup Notebook (Local Version)

**Converted from:** Azure Databricks (Spark + ADLS + MLflow/Databricks)  
**Target:** Local Python + Pandas + joblib  

### Changes from original (53 cells → 12 clean cells):
- **REMOVED:** All `abfss://` paths → local `data/` and `models/` paths
- **REMOVED:** Azure Service Principal / ADLS OAuth config
- **REMOVED:** `spark.read.parquet(...)` → `pd.read_parquet()`
- **REMOVED:** `spark.conf.set(...)`, `dbutils.fs.ls(...)`, `display(...)`
- **REMOVED:** All `pyspark.sql.functions.*`, `pyspark.sql.Window` → Pandas equivalents
- **REMOVED:** `%pip install` magic commands → standard pip (run separately)
- **REMOVED:** Databricks MLflow experiment tracking (`mlflow.set_experiment("/Shared/...")`) → local MLflow
- **REMOVED:** MLflow model registry (Unity Catalog) → local `joblib.dump()`
- **REMOVED:** `spark.sql("SELECT current_catalog()...")` → not needed locally
- **REMOVED:** `mlflow.pyfunc.spark_udf(...)` → direct `model.predict()`
- **REMOVED:** DBFS model save (`/dbfs/FileStore/models/`) → local `models/` directory
- **CONSOLIDATED:** 53 redundant cells (multiple retries, overlapping pip installs, repeated CV) → 12 clean cells

### Input:
- `data/port_daily.parquet` (from ingestion notebook)

### Outputs:
- `data/port_daily_expanded.parquet` — expanded features for ML
- `models/portwatch_lightgbm.pkl` — trained LightGBM model
- `outputs/validation_predictions.parquet` — validation predictions

In [ ]:
# =============================================================================
# Cell 1 — CONFIG & IMPORTS
# =============================================================================
# REMOVED: %pip install commands (run these in your terminal before starting):
#   pip install pandas numpy scikit-learn xgboost lightgbm mlflow joblib pyarrow
# REMOVED: Azure storage account config, abfss:// paths, Service Principal secrets
# REMOVED: spark.conf.set(...) ADLS OAuth configuration
# REMOVED: Databricks MLflow experiment paths (/Shared/...)
# =============================================================================

import os
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

# --- Local path config (replaces abfss:// paths) ---
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "models"
OUTPUTS_DIR = BASE_DIR / "outputs"

# Input (from ingestion notebook)
INPUT_PATH = DATA_DIR / "port_daily.parquet"

# Outputs
EXPANDED_FEATURES_PATH = DATA_DIR / "port_daily_expanded.parquet"
MODEL_SAVE_PATH = MODELS_DIR / "portwatch_lightgbm.pkl"
VAL_PREDS_PATH = OUTPUTS_DIR / "validation_predictions.parquet"

# Ensure directories exist
DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)

print("Paths configured:")
print(f"  Input:      {INPUT_PATH}")
print(f"  Expanded:   {EXPANDED_FEATURES_PATH}")
print(f"  Model:      {MODEL_SAVE_PATH}")
print(f"  Val preds:  {VAL_PREDS_PATH}")

In [ ]:
# =============================================================================
# Cell 2 — LOAD DATA & INSPECT
# =============================================================================
# REPLACED: spark.read.parquet("abfss://features@.../port_daily/")
#           → pd.read_parquet()
# REPLACED: df_spark.printSchema() → print(df.dtypes)
# REPLACED: display(df_spark.limit(20)) → print(df.head(20))
# CONSOLIDATED: Original cells 3-6 (read, OAuth config, re-read, inspect)
# =============================================================================

assert INPUT_PATH.exists(), f"ERROR: {INPUT_PATH} not found. Run the ingestion notebook first."

df = pd.read_parquet(INPUT_PATH)
print(f"Loaded {len(df):,} rows from {INPUT_PATH}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nPreview (first 5 rows):")
print(df.head(5).to_string())

In [ ]:
# =============================================================================
# Cell 3 — VALIDATE SCHEMA & CHECK DATA QUALITY
# =============================================================================
# CONSOLIDATED: Original cells 6-7 (inspect date, find target, null checks, stats)
# REPLACED: pyspark.sql.functions F.min/F.max/F.mean/F.stddev → pandas describe()
# REPLACED: F.sum(F.col(c).isNull().cast("int")) → df.isnull().sum()
# =============================================================================

# Validate expected columns from ingestion notebook
expected_cols = {'portid', 'event_date', 'daily_port_calls', 'lag_1', 'lag_7_avg', 'year', 'month'}
actual_cols = set(df.columns)
missing = expected_cols - actual_cols
if missing:
    raise Exception(f"Missing expected columns: {missing}")
print("✓ All expected columns present")

# Ensure event_date is datetime
df['event_date'] = pd.to_datetime(df['event_date'])
print(f"\nDate range: {df['event_date'].min()} → {df['event_date'].max()}")

# Null counts
print(f"\nNull counts:")
print(df.isnull().sum())

# Target column stats
target = 'daily_port_calls'
print(f"\nTarget '{target}' distribution:")
print(df[target].describe())

# Per-port summary
port_summary = df.groupby('portid')[target].agg(['count', 'mean']).sort_values('count', ascending=False)
print(f"\nTop 20 ports by row count:")
print(port_summary.head(20).to_string())

In [ ]:
# =============================================================================
# Cell 4 — EXPANDED FEATURE ENGINEERING
# =============================================================================
# CONSOLIDATED: Original cell 27 (the big feature expansion cell)
# REPLACED: pyspark.sql.Window → pandas groupby + shift/rolling
# REPLACED: F.lag(...).over(w) → groupby().shift()
# REPLACED: F.avg(...).over(w.rowsBetween(-n, -1)) → groupby().shift(1).rolling(n)
# REPLACED: F.stddev(...).over(w) → groupby().shift(1).rolling(n).std()
# REPLACED: F.dayofweek(...) → dt.dayofweek
# REPLACED: F.coalesce(col, lit(0.0)) → fillna(0.0)
# REPLACED: df.write.parquet(abfss://...) → df.to_parquet(local)
# =============================================================================

print("Starting expanded feature engineering...")

# Sort by port and date (required for correct lag/rolling)
df = df.sort_values(['portid', 'event_date']).reset_index(drop=True)

# --- Lag features ---
lags = [1, 2, 3, 7, 14, 30]
for l in lags:
    df[f'lag_{l}'] = df.groupby('portid')['daily_port_calls'].shift(l).astype('float32')
print(f"Created lag features: {['lag_' + str(l) for l in lags]}")

# --- Rolling window features: mean and std for windows 7, 14, 30 ---
# Original Spark: rowsBetween(-win, -1) = preceding 'win' rows excluding current
for win in [7, 14, 30]:
    df[f'roll_mean_{win}'] = (
        df.groupby('portid')['daily_port_calls']
        .transform(lambda x: x.shift(1).rolling(window=win, min_periods=1).mean()).astype('float32')
    )
    df[f'roll_std_{win}'] = (
        df.groupby('portid')['daily_port_calls']
        .transform(lambda x: x.shift(1).rolling(window=win, min_periods=1).std()).astype('float32')
    )
print("Created rolling mean/std features for windows: [7, 14, 30]")

# --- Day of week and is_weekend ---
# Pandas dayofweek: Monday=0, Tuesday=1, ..., Saturday=5, Sunday=6
df['dow'] = df['event_date'].dt.dayofweek  # Monday=0, Sunday=6
df['is_weekend'] = df['dow'].isin([5, 6]).astype(int)  # Saturday=5, Sunday=6
print("Created dow and is_weekend features")

# --- Port-level static features: running mean and count (excluding current row) ---
df['port_mean_prev'] = (
    df.groupby('portid')['daily_port_calls']
    .transform(lambda x: x.expanding().mean().shift(1)).astype('float32')
)
df['port_count_prev'] = (
    df.groupby('portid')['daily_port_calls']
    .transform(lambda x: x.expanding().count().shift(1)).astype('float32')
)
print("Created port_mean_prev and port_count_prev features")

# --- Fill initial nulls ---
global_mean = df['daily_port_calls'].mean()
df['port_mean_prev'] = df['port_mean_prev'].fillna(global_mean)
df['port_count_prev'] = df['port_count_prev'].fillna(0)

# Fill NA for lag and rolling features with 0
numeric_new_cols = (
    [f'lag_{l}' for l in lags] +
    [f'roll_mean_{w}' for w in [7, 14, 30]] +
    [f'roll_std_{w}' for w in [7, 14, 30]]
)
for c in numeric_new_cols:
    df[c] = df[c].fillna(0.0)

# --- Missing-value indicator features ---
for c in numeric_new_cols:
    df[f'{c}_is_null'] = (df[c] == 0.0).astype(int)
print("Created missing-value indicator features")

# --- Select columns for ML dataset ---
keep_cols = (
    ['event_date', 'portid', 'daily_port_calls', 'year', 'month'] +
    numeric_new_cols +
    [f'{c}_is_null' for c in numeric_new_cols] +
    ['dow', 'is_weekend', 'port_mean_prev', 'port_count_prev']
)
df_ml = df[keep_cols].copy()

print(f"\nExpanded feature set: {df_ml.shape[1]} columns, {len(df_ml):,} rows")
print(f"Feature columns: {[c for c in df_ml.columns if c not in ('event_date','portid','daily_port_calls','year','month')]}")
print(df_ml.head(10).to_string())

In [ ]:
# =============================================================================
# Cell 5 — SAVE EXPANDED FEATURES
# =============================================================================
# REPLACED: df_ml.write.mode("overwrite").partitionBy("year","month").parquet(
#           "abfss://features@.../port_daily_expanded/")
#           → df_ml.to_parquet(local path)
# =============================================================================

print(f"Writing expanded features ({len(df_ml):,} rows) to: {EXPANDED_FEATURES_PATH}")
df_ml.to_parquet(EXPANDED_FEATURES_PATH, index=False)
print(f"✓ Write complete. File size: {EXPANDED_FEATURES_PATH.stat().st_size:,} bytes")

In [ ]:
# =============================================================================
# Cell 6 — PREPARE ML DATA
# =============================================================================
# CONSOLIDATED: Original cells 28-29 (re-read expanded, sample to Pandas)
# REMOVED: Spark sampling (df_exp.sample(False, 0.10, seed=42)) since we're
#          already in Pandas. We use the full dataset directly.
# REMOVED: Spark-to-Pandas conversion (.toPandas()) — already in Pandas
# =============================================================================

# Read back expanded features (ensures clean state)
pdf = pd.read_parquet(EXPANDED_FEATURES_PATH)
print(f"Loaded expanded features: {pdf.shape}")

# Ensure date is datetime and sorted
pdf['event_date'] = pd.to_datetime(pdf['event_date'])
pdf = pdf.sort_values('event_date').reset_index(drop=True)

# Downsample training data to 20% to avoid OOM during LightGBM dataset creation

# Auto-detect feature columns (everything except metadata/target)
NON_FEATURE_COLS = ('event_date', 'portid', 'daily_port_calls', 'year', 'month')
FEATURES = [c for c in pdf.columns if c not in NON_FEATURE_COLS]

print(f"Num features: {len(FEATURES)}")
print(f"Date range: {pdf['event_date'].min()} → {pdf['event_date'].max()}")
print(f"Preview feature columns (first 20): {FEATURES[:20]}")

In [ ]:
# =============================================================================
# Cell 7 — LIGHTGBM ROLLING-ORIGIN CROSS-VALIDATION
# =============================================================================
# CONSOLIDATED: Original cells 30, 34, 36 (3 near-duplicate CV implementations)
# REPLACED: mlflow.set_experiment("/Shared/portwatch_lgb_expanded") → local MLflow
# REPLACED: mlflow.lightgbm.log_model() → local tracking only
# NOTE: MLflow tracking is kept but uses local file store (./mlruns/)
#        If mlflow is not installed, training still works — just no logging.
# =============================================================================

import lightgbm as lgb
from sklearn.metrics import mean_absolute_error
import time

# Try to use MLflow locally (optional)
try:
    import mlflow
    import mlflow.lightgbm
    mlflow.set_experiment("portwatch_lgb_expanded")
    USE_MLFLOW = True
    print("MLflow available — will log runs locally to ./mlruns/")
except ImportError:
    USE_MLFLOW = False
    print("MLflow not installed — training will proceed without experiment tracking")

# --- Fold setup (rolling-origin CV) ---
val_window_days = 30
min_train_days = 180
date_min = pdf['event_date'].min()
date_max = pdf['event_date'].max()

folds = []
current_train_end = date_min + pd.Timedelta(days=min_train_days)
while True:
    val_start = current_train_end + pd.Timedelta(days=1)
    val_end = current_train_end + pd.Timedelta(days=val_window_days)
    if val_end > date_max:
        break
    folds.append((current_train_end, val_start, val_end))
    current_train_end = val_end

print(f"Planned {len(folds)} folds")
for i, (te, vs, ve) in enumerate(folds):
    print(f"  Fold {i+1}: train_end={te.date()} val={vs.date()}→{ve.date()}")

# --- LightGBM parameters ---
params = {
    "objective": "regression",
    "metric": "mae",
    "learning_rate": 0.05,
    "num_leaves": 64,
    "min_data_in_leaf": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.9,
    "bagging_freq": 1,
    "seed": 42,
    "verbose": -1
}

# --- Run CV ---
fold_maes = []
for fold_idx, (train_end, val_start, val_end) in enumerate(folds, 1):
    print(f"\n=== Fold {fold_idx} | train_end {train_end.date()} val {val_start.date()}→{val_end.date()} ===")
    train_mask = pdf['event_date'] <= train_end
    val_mask = (pdf['event_date'] >= val_start) & (pdf['event_date'] <= val_end)
    train_df = pdf.loc[train_mask]
    val_df = pdf.loc[val_mask]
    print(f"Train rows: {len(train_df):,}  Val rows: {len(val_df):,}")

    if len(train_df) < 200 or len(val_df) < 20:
        print("Skipping fold due to insufficient data")
        continue

    X_train = train_df[FEATURES].fillna(0.0)
    y_train = train_df['daily_port_calls']
    X_val = val_df[FEATURES].fillna(0.0)
    y_val = val_df['daily_port_calls']

    lgb_train = lgb.Dataset(X_train, label=y_train)
    lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

    t0 = time.time()
    callbacks = [
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100)
    ]
    bst = lgb.train(
        params, lgb_train, num_boost_round=1000,
        valid_sets=[lgb_train, lgb_val],
        valid_names=['train', 'val'],
        callbacks=callbacks
    )
    t1 = time.time()

    preds = bst.predict(X_val, num_iteration=bst.best_iteration)
    mae = mean_absolute_error(y_val, preds)
    print(f"Fold {fold_idx} MAE: {mae:.4f} | time: {t1-t0:.1f}s")
    fold_maes.append(mae)

    if USE_MLFLOW:
        with mlflow.start_run(nested=True):
            mlflow.log_param("fold_idx", fold_idx)
            mlflow.log_param("train_end", str(train_end.date()))
            mlflow.log_param("val_start", str(val_start.date()))
            mlflow.log_param("val_end", str(val_end.date()))
            mlflow.log_metric("val_mae", float(mae))

# --- CV Summary ---
if len(fold_maes) > 0:
    print(f"\n=== CV Summary ===")
    print(f"Fold MAEs: {[round(m,4) for m in fold_maes]}")
    print(f"Mean MAE: {np.mean(fold_maes):.4f}  Std: {np.std(fold_maes):.4f}")
else:
    print("No folds were run — adjust folding params or check data size.")

In [ ]:
# =============================================================================
# Cell 8 — FULL RETRAIN ON ALL DATA
# =============================================================================
# CONSOLIDATED: Original cells 39-40 (re-read expanded, full retrain)
# REPLACED: mlflow.lightgbm.log_model() → joblib.dump() to local models/
# REMOVED:  spark.read.parquet(expanded_path) → already in memory as pdf
# =============================================================================

import lightgbm as lgb
from sklearn.metrics import mean_absolute_error
import time

print("Starting full retrain...")
print(f"Using {len(pdf):,} rows, {len(FEATURES)} features")

# Use last 30 days as holdout for early stopping
pdf['event_date'] = pd.to_datetime(pdf['event_date'])
max_date = pdf['event_date'].max()
holdout_days = 30
val_start = max_date - pd.Timedelta(days=holdout_days - 1)

train_mask = pdf['event_date'] < val_start
val_mask = pdf['event_date'] >= val_start

train_df = pdf.loc[train_mask].copy()
val_df = pdf.loc[val_mask].copy()
print(f"Train rows: {len(train_df):,}  Val rows (holdout 30d): {len(val_df):,}")

X_train = train_df[FEATURES].fillna(0.0)
y_train = train_df['daily_port_calls']
X_val = val_df[FEATURES].fillna(0.0)
y_val = val_df['daily_port_calls']

lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

params_final = {
    "objective": "regression",
    "metric": "mae",
    "learning_rate": 0.03,
    "num_leaves": 64,
    "min_data_in_leaf": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.9,
    "bagging_freq": 1,
    "seed": 42,
    "verbose": -1
}

t0 = time.time()
callbacks = [lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=100)]
bst_final = lgb.train(
    params_final, lgb_train, num_boost_round=2000,
    valid_sets=[lgb_train, lgb_val],
    valid_names=['train', 'val'],
    callbacks=callbacks
)
t1 = time.time()

# Validation MAE
preds_val = bst_final.predict(X_val, num_iteration=bst_final.best_iteration)
val_mae = mean_absolute_error(y_val, preds_val)
print(f"\nValidation MAE (holdout 30d): {val_mae:.4f}")
print(f"Training time: {t1-t0:.1f}s")
print(f"Best iteration: {bst_final.best_iteration}")

In [ ]:
# =============================================================================
# Cell 9 — SAVE MODEL
# =============================================================================
# REPLACED: joblib.dump(bst, "/dbfs/FileStore/models/portwatch_lightgbm.pkl")
#           → joblib.dump(bst, "models/portwatch_lightgbm.pkl")
# REMOVED: dbutils.fs.ls("dbfs:/FileStore/models/")
# REMOVED: MLflow model registry (Unity Catalog registration)
# =============================================================================

# Save model locally
joblib.dump(bst_final, MODEL_SAVE_PATH)
print(f"✓ Model saved to: {MODEL_SAVE_PATH}")
print(f"  File size: {MODEL_SAVE_PATH.stat().st_size:,} bytes")

# Also save the feature list for downstream notebooks
feature_list_path = MODELS_DIR / "feature_columns.txt"
with open(feature_list_path, 'w') as f:
    f.write('\n'.join(FEATURES))
print(f"✓ Feature list saved to: {feature_list_path} ({len(FEATURES)} features)")

# Verify model can be loaded
loaded_model = joblib.load(MODEL_SAVE_PATH)
print(f"✓ Model reload verified: {type(loaded_model)}")

In [ ]:
# =============================================================================
# Cell 10 — FULL-BATCH PREDICTION
# =============================================================================
# CONSOLIDATED: Original cells 44-45 (Spark UDF prediction + write)
# REPLACED: mlflow.pyfunc.spark_udf(spark, model_uri) → direct model.predict()
# REPLACED: F.coalesce(F.col(c), F.lit(0.0)) → fillna(0.0)
# REPLACED: F.round(F.when(pred < 0, 0).otherwise(pred), 3) → clip + round
# REPLACED: out_df.write.parquet("abfss://predictions@...") → to_parquet(local)
# =============================================================================

print("Running full-batch prediction...")

# Predict on all data
X_all = pdf[FEATURES].fillna(0.0)
pdf['pred_daily_port_calls'] = bst_final.predict(X_all, num_iteration=bst_final.best_iteration)

# Clip negative predictions to 0 and round
pdf['pred_daily_port_calls'] = pdf['pred_daily_port_calls'].clip(lower=0).round(3)

# Select output columns
out_df = pdf[['event_date', 'portid', 'daily_port_calls', 'pred_daily_port_calls', 'year', 'month']].copy()

print(f"Predictions generated: {len(out_df):,} rows")
print(f"\nSample predictions (first 20):")
print(out_df.sort_values(['event_date', 'portid']).head(20).to_string())

In [ ]:
# =============================================================================
# Cell 11 — SAVE VALIDATION PREDICTIONS
# =============================================================================
# REPLACED: spark_pred.write.mode("overwrite").partitionBy("year","month")
#           .parquet("abfss://predictions@.../daily_port_calls/")
#           → df.to_parquet(local)
# =============================================================================

print(f"Writing predictions ({len(out_df):,} rows) to: {VAL_PREDS_PATH}")
out_df.to_parquet(VAL_PREDS_PATH, index=False)
print(f"✓ Write complete. File size: {VAL_PREDS_PATH.stat().st_size:,} bytes")

In [ ]:
# =============================================================================
# Cell 12 — VALIDATE & SUMMARIZE
# =============================================================================
# CONSOLIDATED: Original cells 45-46 (readback + MAE computation)
# REPLACED: spark.read.parquet(pred_path) → pd.read_parquet()
# =============================================================================

print("=== Final Validation ===")

# Read back predictions
val_readback = pd.read_parquet(VAL_PREDS_PATH)
print(f"Readback: {len(val_readback):,} rows")
print(f"Columns: {list(val_readback.columns)}")

# Compute MAE on holdout (last 30 days)
val_readback['event_date'] = pd.to_datetime(val_readback['event_date'])
max_date = val_readback['event_date'].max()
val_start = max_date - pd.Timedelta(days=29)
holdout = val_readback[val_readback['event_date'] >= val_start]

mae = mean_absolute_error(holdout['daily_port_calls'], holdout['pred_daily_port_calls'])
print(f"\nHoldout MAE (last 30 days): {mae:.4f}")
print(f"Holdout rows: {len(holdout):,}")

# Show sample
print(f"\nSample predictions vs actuals:")
print(holdout[['event_date', 'portid', 'daily_port_calls', 'pred_daily_port_calls']].head(20).to_string())

# Summary of all outputs
print(f"\n{'='*60}")
print(f"NOTEBOOK COMPLETE — Output summary:")
print(f"{'='*60}")
print(f"  1. {EXPANDED_FEATURES_PATH} ({EXPANDED_FEATURES_PATH.stat().st_size:,} bytes)")
print(f"  2. {MODEL_SAVE_PATH} ({MODEL_SAVE_PATH.stat().st_size:,} bytes)")
print(f"  3. {VAL_PREDS_PATH} ({VAL_PREDS_PATH.stat().st_size:,} bytes)")
print(f"  4. {MODELS_DIR / 'feature_columns.txt'}")